# 🎮 游戏助手 LoRA 微调 v2 (7B + 工具调用)
# 基于 Qwen2.5-7B-Instruct + unsloth QLoRA
#
# 操作步骤：
# 1. 菜单栏「代码执行程序」→「更改运行时类型」→ 选 T4 GPU
# 2. 上传 game_assistant_v2.jsonl 到 Colab（聊天+工具调用混合数据）
# 3. 按顺序 Shift+Enter 跑每一格

In [ ]:
# ============================================================
# 第 1 步：挂载 Google Drive（断线保命！）
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/game_assistant_lora"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"[OK] 检查点目录: {SAVE_DIR}")

In [ ]:
# ============================================================
# 第 2 步：安装依赖 + 加载 7B 模型（~5 分钟）
# ============================================================
!pip install -q unsloth transformers datasets peft accelerate
!pip install -q "trl>=0.18.2,<0.25"

from unsloth import FastLanguageModel
import torch

# Qwen2.5-7B-Instruct + QLoRA 4bit ≈ 4GB 显存，T4 16GB 绰绰有余
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,    # 加长序列以容纳工具调用
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可训练参数: {trainable/1e6:.1f}M / {total/1e9:.1f}B ({trainable/total*100:.1f}%)")

In [ ]:
# ============================================================
# 第 3 步：上传训练数据
# ============================================================
from google.colab import files
print("请上传 game_assistant_v2.jsonl 文件...")
uploaded = files.upload()

print("\n上传完成！")

In [ ]:
# ============================================================
# 第 4 步：训练 → 合并 → 转 GGUF
# ============================================================
import json
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# ---- 4a. 加载数据 ----
data = []
with open("game_assistant_v2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))
print(f"加载了 {len(data)} 条训练数据")

# 格式化：处理 role=user / assistant / tool / tool_call
def format_chat(example):
    text = ""
    for msg in example["messages"]:
        role = msg["role"]
        content = msg.get("content", "")
        if role == "user":
            text += f"<|im_start|>user\n{content}<|im_end|>\n"
        elif role == "assistant":
            if msg.get("tool_calls"):
                # 工具调用格式
                tc_json = json.dumps(msg["tool_calls"], ensure_ascii=False)
                text += f"<|im_start|>assistant\n{tc_json}<|im_end|>\n"
            else:
                text += f"<|im_start|>assistant\n{content}<|im_end|>\n"
        elif role == "tool":
            text += f"<|im_start|>tool\n{content}<|im_end|>\n"
    return {"text": text}

dataset = Dataset.from_list(data).map(format_chat)
print(f"数据格式化完成，开始训练...")

# ---- 4b. 训练（7B 用更小 batch） ----
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=2048,
        per_device_train_batch_size=2,       # 7B 模型减小 batch
        gradient_accumulation_steps=8,       # 等效 batch=16
        num_train_epochs=3,
        learning_rate=1e-4,                  # 7B 用更低学习率
        warmup_steps=10,
        fp16=True,
        logging_steps=5,
        save_steps=99999,
        output_dir=SAVE_DIR,
        report_to="none",
    ),
)

print("开始训练...")
trainer.train()
print("[OK] 训练完成！")

# ---- 4c. 合并权重 ----
print("合并 LoRA 权重...")
model.save_pretrained_merged(
    f"{SAVE_DIR}/merged_model_v2",
    tokenizer,
    save_method="merged_16bit",
)
print(f"[OK] 合并模型已保存到 Drive: {SAVE_DIR}/merged_model_v2")

# ---- 4d. 转换 GGUF ----
print("转换 GGUF (7B q8_0 ≈ 7GB)...")
!git clone -q https://github.com/ggerganov/llama.cpp.git 2>/dev/null
!pip install -q -r llama.cpp/requirements.txt 2>/dev/null

gguf_path = f"{SAVE_DIR}/game_assistant_7b.gguf"
!python llama.cpp/convert_hf_to_gguf.py {SAVE_DIR}/merged_model_v2 \
    --outtype q4_k_m \
    --outfile {gguf_path}

print(f"\n[OK] GGUF 已保存到 Drive: {gguf_path}")
print(f"文件大小: {__import__('os').path.getsize(gguf_path) / 1e9:.2f} GB")

In [ ]:
# (此步骤已合并到第 4 步，跳过)

In [ ]:
# (此步骤已合并到第 4 步，跳过)

In [ ]:
# ============================================================
# 第 5 步：下载 GGUF
# ============================================================
from google.colab import files
files.download(f"{SAVE_DIR}/game_assistant_7b.gguf")

## 🎯 导入 Ollama

下载 `game_assistant_7b.gguf` 后，在本地终端：

```
cd 下载目录

ollama create game-assistant-v2 -f Modelfile
```

Modelfile:
```
FROM ./game_assistant_7b.gguf
TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>"""
PARAMETER temperature 0.8
PARAMETER top_p 0.9
```

然后把用户 profile 的 model 改为 `game-assistant-v2`。